# Data Processing
This notebook demonstrates how different datasets can be processed to create a train or fine tuning dataset for LLM training. 

## Core Objectives:
- Demonstrate knowledge distillation
- Demonstrate CoT distillation
- Demonstrate how to combine or restructure datasets for custom use case
- Demonstrate instruction tuning
- Push the dataset into **huggingface hub**

### Knowledge Distillation
In the LLM world, Knowledge Distillation means taking a massive, highly capable "Teacher Model" (like the 109B Llama 4 Scout) and using it to generate data that trains a smaller, cheaper "Student Model" (like 4B Qwen).

### CoT Distillation
When ee are specifically asking the Teacher to generate the step-by-step logic (for example a 3-sentence explanation or a short summary of reasoning), we are performing CoT Distillation. We aren"t just teaching the student what the answer is; we are distilling the teacher"s reasoning process into the student"s weights. 

### Instruction Tuning
Other modifications made to the dataset for example formatting it into strict JSON, injecting a specific schema, or forcing the model to follow precise, specific instructions is called Instruction Tuning (or Data Alignment).

W are basically designing a strict "Instruction Format" that the model must follow.

### MedQA Dataset Processing
Process the dataset to create the following schema for every row of data. Since the dataset already has all the information

```json
{
    "question": "<USER_QUESTION>",
    "options": {
        "option_a": "<OPTION_A>",
        "option_b": "<OPTION_B>",
        "option_b": "<OPTION_C>",
        "option_b": "<OPTION_D>",
    },
    "reasoning": "<REASONING>",
    "correct_option": "<CORRECT_OPTION>"
}
```

### GSM8K Dataset Processing
Process the dataset into the same schema as the `MedQA` dataset. But since the `gsm8k` is not MCQ by default, we need to manually structure this into the required format. This can be done with standard data processing and NLP techniques (like regex based extraction)

In [6]:
import json
import random
import pandas as pd
from datasets import load_dataset
from groq import Groq
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

# =================================
# CONFIGURATION
# =================================
client = Groq()

MODEL_NAME = "meta-llama/llama-4-scout-17b-16e-instruct"
MEDQA_TOTAL_SIZE = 2000
GSM8K_TOTAL_SIZE = 500

OUTPUT_FILE = "medqa-gsm8k-modified-v1.jsonl"

random.seed(42)

# =================================
# MEDQA KNOWLEDGE DISTILLATION
# =================================
def get_reasoning(question, options, correct_answer):
    system_prompt = (
        "Expert Board Examiner. Analyze the medical case and write a concise, "
        "3-sentence factual explanation for the correct answer. Explain why the "
        "primary distractor is less likely. Output ONLY the paragraph. No filler."
    )
    user_prompt = f"Q: {question}\nOptions: {options}\nCorrect Answer: {correct_answer}"

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1,
            max_tokens=512
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

# =================================
# MEDQA STRATIFIED SAMPLING
# =================================
def process_medqa_stratified():
    print("\033[1;36mLoading and cleaning MedQA dataset...\033[0m")

    # Load the dataset into dataframe
    ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")
    df = ds.to_pandas()

    # Map answer to A/B/C/D labels
    def get_label(row):
        if "answer_idx" in row and pd.notna(row["answer_idx"]):
            return row["answer_idx"]
        opts = row["options"]
        ans = row["answer"]
        found = [k for k, v in opts.items() if str(v).strip() == str(ans).strip()]
        return found[0] if found else "A"
    
    df["label"] = df.apply(get_label, axis=1)

    print("\033[1;36mPerforming stratified sampling (500 rows per label)...\033[0m")

     # Stratified sampling
    valid_df = df[df["label"].isin(["A", "B", "C", "D"])]
    stratified_df = valid_df.groupby("label").sample(n=500, random_state=42).reset_index(drop=True)

    final_data = []
    for _, row in tqdm(stratified_df.iterrows(), total=MEDQA_TOTAL_SIZE, desc="Distilling Reasoning"):
        # The options are already a dictionary in this dataset schema
        opts = row["options"] 
        question = row["question"]
        correct_ans_text = opts.get(row["label"], "")

        # Distill reasoning using llama-4-scout
        reasoning = get_reasoning(question, opts, correct_ans_text)

        final_data.append({
            "question": question,
            "options": opts,
            "reasoning": reasoning,
            "correct_option": row["label"]
        })

    return final_data

# =================================
# GSM8K TRANSFORMATION
# =================================
def process_gsm8k():
    print("\033[1;36mLoading gsm8k for transformation...\033[0m")

    # Load dataset into dataframe
    ds = load_dataset("openai/gsm8k", "main", split="train")
    df = ds.to_pandas().sample(n=GSM8K_TOTAL_SIZE, random_state=42)

    final_data = []
    for _, row in df.iterrows():
        parts = row["answer"].split("####")
        reasoning = parts[0].strip()
        final_val = parts[1].strip()

        # Generate dummy values for schema alignment
        num = int(final_val.replace(",", ""))
        distractors = [
            str(num + random.randint(1, 5)),
            str(num - 2),
            str(num * 2)
        ]
        opts_list = distractors + [final_val]
        random.shuffle(opts_list)

        opts_dict = {chr(65+i): v for i, v in enumerate(opts_list)}
        correct_letter = [k for k, v in opts_dict.items() if v == final_val][0]

        final_data.append({
            "question": row["question"],
            "options": opts_dict,
            "reasoning": reasoning,
            "correct_option": correct_letter
        })
    
    return final_data

In [7]:
from datasets import Dataset

# =================================
# EXECUTION
# =================================
medqa_rows = process_medqa_stratified() # medical reasoning of medqa
gsm8k_rows = process_gsm8k() # gsm8k re-structuring based on schema

# Combine and shuffle
train_data = medqa_rows + gsm8k_rows
random.shuffle(train_data)

print(f"\033[1;36m-> Dataset compiled successfully with {len(train_data)} rows...\033[0m")
print(f"\033[1;36m-> Pushing to HuggingFace hub...\033[0m")

try:
    # Convert the list of dictionaries directly into a HuggingFace dataset
    hf_dataset = Dataset.from_list(train_data)

    # Push to hub
    hf_dataset.push_to_hub(
        repo_id="loknezmonzter/medqa-gsm8k-modified-v1"
    )
    print("✅ SUCCESS: Dataset pushed to Hugging Face Hub!")

except Exception as e:
    print(f"❌ ERROR Failed to push to Hugging Face: {e}")

    # Fallback: Write locally if HF upload fails
    fallback_file = "iqvia_slm_train_fallback.jsonl"
    print(f"Executing fallback: Saving locally to {fallback_file}")
    with open(fallback_file, "w", encoding="utf-8") as f:
        for item in train_data:
            f.write(json.dumps(item) + "\n")

Loading and cleaning MedQA dataset...


Performing stratified sampling (500 rows per label)...


Distilling Reasoning: 100%|██████████| 2000/2000 [48:10<00:00,  1.45s/it] 


Loading gsm8k for transformation...


README.md: 0.00B [00:00, ?B/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ SUCCESS: Dataset pushed to Hugging Face Hub!


In [5]:
import os
import json
from openai import OpenAI
from typing import Dict, Any

# ==========================================
# Configuration & Client Initialization
# ==========================================
# Pointing to the local vLLM server hosted on the A6000
VLLM_API_BASE = "http://172.17.0.1:8000/v1"
MODEL_ID = "unsloth/medgemma-27b-text-it-bnb-4bit"

client = OpenAI(
    base_url=VLLM_API_BASE,
    api_key="vllm-local" # API key is required by the client but ignored by local vLLM
)

# ==========================================
# Schema & Prompt Definition
# ==========================================
# We define the JSON schema strictly within the System Prompt.
# vLLM's guided decoding will force the model's output to conform to this structure.

SYSTEM_PROMPT = """You are an expert clinical informatician. Your task is to extract highly structured knowledge from raw clinical text to train smaller models. 

You must analyze the text and return a strict JSON object following this EXACT schema:
{
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}

Output ONLY valid JSON. Do not include markdown formatting like ```json."""

In [6]:
# ==========================================
# Extraction Logic
# ==========================================
def extract_clinical_data(clinical_text: str):
    """
    Sends clinical text to hosted llm via vLLM and retrieves the structured JSON.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Extract structured clinical data from the following text:\n\n{clinical_text}"}
            ],  
            max_tokens=4096, 
            top_p=0.9,
            response_format={"type": "json_object"}
        )
        
        # Parse the JSON string returned by the model
        raw_output = response.choices[0].message.content
        structured_data = json.loads(raw_output)

        print(f"FULL_OUTPUT\n\n{response}\n{'='*50}\n\n")
        return structured_data

    except json.JSONDecodeError as e:
        print(f"[Error] Failed to parse JSON. Model output might be malformed: {e}")
        return {"error": "JSON Parse Error", "raw_output": raw_output}
    except Exception as e:
        print(f"[Error] API Call Failed: {e}")
        return {"error": str(e)}

In [7]:
import re

def clean_clinical_text(text):
    # This range [\u0100-\u01ff] captures all Byte-level BPE artifacts
    # We replace them with their standard ASCII equivalents (Shifted by 256)
    return re.sub(r'[\u0100-\u01ff]', lambda m: chr(ord(m.group(0)) - 256), text)
# ==========================================
# Execution & Testing
# ==========================================
sample_patient_text = """
A 45-year-old male presented to the ER with severe, crushing chest pain radiating to the left arm. 
ECG showed ST-elevation in leads V2-V4. The patient denied any history of smoking or diabetes. 
He was immediately administered 300mg of Aspirin and rushed to the cath lab for suspected anterior STEMI.
"""

print("Sending request to vllm...")

data = extract_clinical_data(sample_patient_text)

print("\n--- Extracted JSON Output ---")
print(json.dumps(data, indent=4))

Sending request to vllm...


FULL_OUTPUT

ChatCompletion(id='chatcmpl-8100e25a083b44b4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"summary": "A 45-year-old male presented with symptoms suggestive of an acute myocardial infarction, specifically an anterior STEMI, based on chest pain characteristics and ECG findings. He received immediate treatment with Aspirin and underwent cardiac catheterization.", "clinical_reasoning": "1. The patient presents with classic symptoms of acute myocardial infarction (AMI): severe, crushing chest pain radiating to the left arm. 2. The ECG confirms an ST-elevation myocardial infarction (STEMI) in the anterior wall (leads V2-V4). 3. Based on the symptoms and ECG findings, a diagnosis of suspected anterior STEMI is made. 4. Immediate treatment with Aspirin is initiated as standard protocol for AMI. 5. The patient is sent to the cath lab for definitive diagnosis and treatment (e.g., PCI) due to the high suspicion of STEMI.", "r

In [2]:
!curl http://172.17.0.1:8000/v1/models

{"object":"list","data":[{"id":"unsloth/medgemma-27b-text-it-bnb-4bit","object":"model","created":1779297292,"owned_by":"vllm","root":"unsloth/medgemma-27b-text-it-bnb-4bit","parent":null,"max_model_len":16384,"permission":[{"id":"modelperm-a79afda22cdf71e2","object":"model_permission","created":1779297292,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [11]:
!ls -lh session_logs.txt

ls: cannot access 'session_logs.txt': No such file or directory


In [1]:
print("hello")

hello
